In [ ]:
#import uproot
import awkward as ak

import pandas as pd

import matplotlib.pylab as plt
import numpy as np

import hist
from hist import Hist

import time

import myPIDselector

import math

import os

data = ak.from_parquet("Background_and_signal_SP_modes_All_runs.parquet")

B_D1_IDX_FIELD            = 'Bd1Idx'
B_D2_IDX_FIELD            = 'Bd2Idx'

# Lambda daughter fields
LAMBDA_PROTON_IDX_FIELD   = 'Lambda0d1Idx'
LAMBDA_PROTON_LUND_FIELD  = 'Lambda0d1Lund'

 
PROTON_TRKIDX_FIELD       = 'pTrkIdx'
PROTON_SELECTORS_FIELD    = 'pSelectorsMap'
TRK_LUND_FIELD            = 'TRKLund'

MES_FIELD                 = 'BpostFitMes'
DELTAE_FIELD              = 'BpostFitDeltaE'

SPMODE_FIELD              = 'spmode'
plt.close()

def calculate_bits_for_PID_selector(trkidx, trk_selector_map, verbose=0):
    
    bits = None

    # If there is no trk index passed in, just calculate the bits for
    # all of the tracks
    if trkidx is None:
        trkidx = ak.local_index(trk_selector_map)

    # Grab the tracks that map on to the particle/collection we are interested in 
    subset_of_trk_selector_map = trk_selector_map[trkidx]
    if verbose:
        print("values in the subset of the trk selector map")
        print(subset_of_trk_selector_map)
        print()
        
    # We need this to convert the numbers in the selector to binary
    binary_repr_vec = np.vectorize(np.binary_repr)

    # Grab the number of entries in each so we can unflatten this later
    counts = ak.num(subset_of_trk_selector_map)
    
    # Now get the binary representation (as a string) for the flattened subset
    binrep = binary_repr_vec(ak.flatten(subset_of_trk_selector_map), width=32)

    if verbose:
        print("binary representation of selector map")
        print(binrep)
        print()

    # Convert the string to integers
    tempbits = np.array(binrep).astype(int)
    bits = ak.unflatten(tempbits,counts)

    if verbose:
        print("flattened integer representation of selectors as binary (int)")
        print(tempbits)
        print()
        print("unflattened integer representation of selectors as binary (int)")
        print(bits)
        print()

    return bits
def mask_PID_selection(bits, selector, pid_map_object):


    bit_to_look_for = pid_map_object.selectors.index(selector)

    place = int(math.pow(10,bit_to_look_for))
    #print(place)

    mask = bits // place % 10

    mask_bool = ak.values_astype(mask,bool)

    return mask_bool
def get_info_for_PID_masks(ak_arr, verbosity=0):

    # Proton and pion information from the Lambda decay
    # These are the index of the proton (d1) and pion (d2) in those lists
    L1d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd1Idx]
    L1d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd1Idx]

    L2d1idx = ak_arr.Lambda0d1Idx[ak_arr.Bd2Idx]
    L2d2idx = ak_arr.Lambda0d2Idx[ak_arr.Bd2Idx]


    
     
    #d1lund = ak_arr['Lambda0d1Lund']
    #d2lund = ak_arr['Lambda0d2Lund']
    
    #Bd2idx = ak_arr['Bd2Idx']
    #Bd2lund = ak_arr['Bd2Lund']
    
    trkidx_proton = ak_arr['pTrkIdx']
    trk_selector_map_proton = ak_arr['pSelectorsMap']
    
    trkidx_pion = ak_arr['piTrkIdx']
    trk_selector_map_pion = ak_arr['piSelectorsMap']

    indices_and_maps = {}
    indices_and_maps['L1d1idx'] = L1d1idx
    indices_and_maps['L1d2idx'] = L1d2idx
    indices_and_maps['L2d1idx'] = L2d1idx
    indices_and_maps['L2d2idx'] = L2d2idx
    #indices_and_maps['Bd2idx'] = Bd2idx

    indices_and_maps['trkidx_proton'] = trkidx_proton
    indices_and_maps['trk_selector_map_proton'] = trk_selector_map_proton

    indices_and_maps['trkidx_pion'] = trkidx_pion
    indices_and_maps['trk_selector_map_pion'] = trk_selector_map_pion

    return indices_and_maps
def PID_masks(indices_and_maps, \
              lam1p_selector,\
              lam1pi_selector, \
              lam2p_selector,\
              lam2pi_selector, \
             verbosity=0):
                #change Bpselector
    L1d1idx = indices_and_maps['L1d1idx']
    L1d2idx = indices_and_maps['L1d2idx']
    L2d1idx = indices_and_maps['L2d1idx']
    L2d2idx = indices_and_maps['L2d2idx']
    #Bd2idx = indices_and_maps['Bd2idx']

    trkidx_proton = indices_and_maps['trkidx_proton']
    trk_selector_map_proton = indices_and_maps['trk_selector_map_proton']

    trkidx_pion = indices_and_maps['trkidx_pion']
    trk_selector_map_pion = indices_and_maps['trk_selector_map_pion']
    
    # Proton
    pbits = calculate_bits_for_PID_selector(trkidx_proton, trk_selector_map_proton, verbose=verbosity)
    # Pion
    pibits = calculate_bits_for_PID_selector(trkidx_pion, trk_selector_map_pion, verbose=verbosity)
    
    
    mask_bool_protonL1 = mask_PID_selection(pbits[L1d1idx], lam1p_selector, pps)
        
    mask_bool_pionL1 = mask_PID_selection(pibits[L1d2idx], lam1pi_selector, pips)

    mask_bool_protonL2 = mask_PID_selection(pbits[L2d1idx], lam2p_selector, pps)
        
    mask_bool_pionL2 = mask_PID_selection(pibits[L2d2idx], lam2pi_selector, pips)

    return mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2
def indices_to_booleans(indices, array_to_slice):
    """Convert ragged integer indices to a boolean mask over array_to_slice."""
    whole_set, in_set = ak.unzip(ak.cartesian(
        [ak.local_index(array_to_slice), indices], nested=True
    ))
    return ak.any(whole_set == in_set, axis=-1)
def ragged_element_pick(ragged_arr, flat_idx):
    """
    For each event i, return ragged_arr[i][flat_idx[i]].

    Parameters
    ----------
    ragged_arr : ak.Array, shape [n_events, var]
    flat_idx   : array-like, shape [n_events]  (integer index per event)

    Returns
    -------
    np.ndarray, shape [n_events]
    """
    flat_content = ak.flatten(ragged_arr)
    counts       = ak.num(ragged_arr)

    offsets = np.zeros(len(counts) + 1, dtype=np.int64)
    np.cumsum(ak.to_numpy(counts), out=offsets[1:])

    idx_np = ak.to_numpy(flat_idx).astype(np.int64)
    global_indices = offsets[:-1] + idx_np

    return ak.to_numpy(flat_content)[global_indices]
def build_antilambda_antimask(ak_arr, pps,
                               selector,
                               verbose=0):
    """
    Antibaryon veto for B0 -> Lambda Lambda (BNV).

    REQUIRES: single-B-candidate cut already applied (exactly one B per event).

    Returns
    -------
    charge_test : awkward array of booleans
        Per-track flag — True where a rest-of-event track is an antiproton.
    """

    n = len(ak_arr)

     # 1.  Squeeze B-candidate dimension
    #bd1idx = ak.to_numpy(ak_arr[B_D1_IDX_FIELD][:, 0])   # scalar per event
    bd1idx = ak.to_numpy(ak.firsts(ak_arr[B_D1_IDX_FIELD], axis=1))

    #bd2idx = ak.to_numpy(ak_arr[B_D2_IDX_FIELD][:, 0])
    bd2idx = ak.to_numpy(ak.firsts(ak_arr[B_D2_IDX_FIELD], axis=1))

    if verbose:
        print(f"bd1idx  {bd1idx[:5]}")
        print(f"bd2idx  {bd2idx[:5]}")
        print()

     
    # 2.  Chain: B daughter Lambda index -> that Lambda's proton index
     
    all_lam_proton_idx  = ak_arr[LAMBDA_PROTON_IDX_FIELD]
    all_lam_proton_lund = ak_arr[LAMBDA_PROTON_LUND_FIELD]

    # Proton-list index for each B-daughter Lambda (flat numpy, one per event)
    lam1_pidx = ragged_element_pick(all_lam_proton_idx, bd1idx)
    lam2_pidx = ragged_element_pick(all_lam_proton_idx, bd2idx)

    if verbose:
        print(f"lam1_pidx (proton-list index from Λ₁)  {lam1_pidx[:5]}")
        print(f"lam2_pidx (proton-list index from Λ₂)  {lam2_pidx[:5]}")
        print()

     #     qlam1 = +1 for Λ → p π⁻,  -1 for Λ̄ → p̄ π⁺
    lam1_lund = ragged_element_pick(all_lam_proton_lund, bd1idx)
    lam2_lund = ragged_element_pick(all_lam_proton_lund, bd2idx)

    qlam1 = np.sign(lam1_lund).astype(float)   # shapes
    qlam2 = np.sign(lam2_lund).astype(float)

    if verbose:
        print(f"qlam1 (charge of Λ₁ proton)  {qlam1[:5]}")
        print(f"qlam2 (charge of Λ₂ proton)  {qlam2[:5]}")
        print()

    qtrk = ak_arr[TRK_LUND_FIELD] / np.abs(ak_arr[TRK_LUND_FIELD])

     # 4.  Map proton-list indices to track-list indices
    trk_selector_map = ak_arr[PROTON_SELECTORS_FIELD]
    trkidx_proton    = ak_arr[PROTON_TRKIDX_FIELD]

    # Pick the track index for each signal proton (flat numpy)
    lam1_trkidx_flat = ragged_element_pick(trkidx_proton, lam1_pidx)
    lam2_trkidx_flat = ragged_element_pick(trkidx_proton, lam2_pidx)

    # Wrap as length-1 ragged for indices_to_booleans
    lam1_trkidx = ak.unflatten(lam1_trkidx_flat, 1)
    lam2_trkidx = ak.unflatten(lam2_trkidx_flat, 1)

    if verbose:
        print(f"lam1_trkidx  {lam1_trkidx[:5]}")
        print(f"lam2_trkidx  {lam2_trkidx[:5]}")
        print()

    bool_idx1 = indices_to_booleans(lam1_trkidx, trk_selector_map)
    bool_idx2 = indices_to_booleans(lam2_trkidx, trk_selector_map)

    if verbose:
        print(f"bool_idx1 (Λ₁ proton)  {bool_idx1[:3]}")
        print(f"bool_idx2 (Λ₂ proton)  {bool_idx2[:3]}")
        print()

    
    # 5.  PID selection on ALL proton candidates  
    pbits     = calculate_bits_for_PID_selector(None, trk_selector_map)
    mask_bool = mask_PID_selection(pbits, selector, pps)

    if verbose:
        print(f"Tracks passing {selector}:  {ak.sum(ak.num(pbits[mask_bool]))}")
        print()

     # 6.  Charge test on REST-OF-EVENT proton candidates
     
    other_mask  = ~(bool_idx1 | bool_idx2) & mask_bool
    charge_test = qtrk[other_mask] == -qlam1

    if verbose:
        print(f"charge_test  {charge_test[:3]}")

   
    have_opp       = ak.num(charge_test[charge_test])
    nhave_opp      = len(have_opp[have_opp > 0])
    ndont_have_opp = len(have_opp[have_opp == 0])

    return charge_test
def apply_antibaryon_veto(ak_arr, pps, selector):
    """Return a per-event boolean: True = passes veto (no antiproton found)."""
    ct       = build_antilambda_antimask(ak_arr, pps, selector=selector, verbose=0)
    have_opp = ak.num(ct[ct])
    return have_opp == 0

pps = myPIDselector.PIDselector("p")
pips = myPIDselector.PIDselector("pi")

pidChoiceDict = {}

myPMask = "SuperLooseKMProtonSelection"
myPiMask = "SuperLooseKMPionMicroSelection"
    


""" 
1. BcosthCM
2. BCosThetaS
3. BCosThetaT
4. BLegendreP2
5. thrustMagAll
6. R2All

"""

    
maxSigCut = 10
minSigCut = 10

BM = Hist.new.Reg(50, -1.25, 1.25, name="BcosthCM", label=r"BcosthCM") \
        .StrCat([], name="SP1", label="SP modes", growth=True)\
        .Weight()

BS = Hist.new.Reg(50, -1.25, 1.25, name="BCosThetaS", label=r"BCosThetaS") \
        .StrCat([], name="SP2", label="SP modes", growth=True)\
        .Weight()

BT = Hist.new.Reg(50, -1.25, 1.25, name="BCosThetaT", label=r"BCosThetaT") \
        .StrCat([], name="SP3", label="SP modes", growth=True)\
        .Weight()

B2 = Hist.new.Reg(50, -1, 7.5, name="BLegendreP2", label=r"BLegendreP2") \
        .StrCat([], name="SP4", label="SP modes", growth=True)\
        .Weight()

TA = Hist.new.Reg(50, 0.55, 1.05, name="thrustMagAll", label=r"thrustMagAll") \
        .StrCat([], name="SP5", label="SP modes", growth=True)\
        .Weight()

RA = Hist.new.Reg(50, -0.1, 1, name="R2All", label=r"R2All") \
        .StrCat([], name="SP6", label="SP modes", growth=True)\
        .Weight()
x1 = np.array([])
y1 = np.array([])
x2 = np.array([])
y2 = np.array([])
x1S = np.array([])
y1S = np.array([])
x2S = np.array([])
y2S = np.array([])



val998 = []
val1005 = []
val999 = []


#b = Hist.new.Reg(50, -1, 7.5, name="b", label=f"BLegendreP2, thrustMagAll<{i/100}") \
    #.StrCat([], name="SP7", label="SP modes", growth=True)\
    #.Weight()
if False:
    for spmode in ['998','1005','-999']:
        spmask = (data['spmode']==spmode)
        indices_and_maps = get_info_for_PID_masks(data, verbosity=0)

        mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                    lam1p_selector=myPMask, \
                    lam1pi_selector=myPiMask, \
                    lam2p_selector=myPMask, \
                    lam2pi_selector=myPiMask, \
                    verbosity=0)
        mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2 
        mask_fit = mask_pid[spmask]
        lamconflsig = data[spmask]['Lambda0postFitFlightSignificance'][mask_fit]
        numBCan = data[spmask]['nB']
        mask_fl = (lamconflsig>10)
        flightSigMask = (ak.all(mask_fl,axis=1))
        bCanMask = (numBCan==1)
        vetoMask = apply_antibaryon_veto(data, pps, myPMask)
        vetoMask = vetoMask[spmask]

        BCTT = data[spmask]['BCosThetaT']
        BCTT = BCTT[flightSigMask&bCanMask&vetoMask]

        BCTS = data[spmask]['BCosThetaS']
        BCTS = BCTS[flightSigMask&bCanMask&vetoMask]

        TMA = data[spmask]['thrustMagAll']
        TMA = TMA[flightSigMask&bCanMask&vetoMask]

        BP2 = data[spmask]['BLegendreP2']
        BP2 = BP2[flightSigMask&bCanMask&vetoMask]

        
            #bcttMask = (BCTT<i/50)
            #BCTSmasked = BCTS[bcttMask]
        for i in range(10):
            for j in range(10):
                for p in range(10):
                    for q in range(10):
                        tmaMask = (TMA<(i+79)/100)
                        bp2Mask = (BP2<(j/10)+1.2)
                        bcttMask = (BCTT<(p/100)+0.47)
                        bctsMask = (BCTS<(q/100)+0.58)


                        BP2masked = BP2[bcttMask&bctsMask&bp2Mask]
                        BP2masked = BP2masked[tmaMask]

                        

                        if spmode == '998':
                            val998 += [ak.sum(ak.num(BP2masked))]
                        elif spmode == '1005':
                            val1005 += [ak.sum(ak.num(BP2masked))]
                        elif spmode == '-999':
                            val999 += [ak.sum(ak.num(BP2masked))]

                        #x = ak.to_numpy(ak.flatten(BP2masked))
                        #x = ak.to_numpy(TMAmasked)

                        #if spmode == '998':
                        #    b.fill(b=x, SP7=spmode,  weight=(424.3/1719))
                        #elif spmode == '1005':
                        #    b.fill(b=x, SP7=spmode,  weight=(424.3/868.3))
                        #elif spmode == '-999':
                        #    b.fill(b=x, SP7=spmode,  weight=1)

            


        #plt.figure(figsize=(8,6))
        #b.stack('SP7')[:].project("b").plot(stack=True,histtype="fill")
        #plt.legend() 
        #plt.show()

    denom998 = ak.sum(ak.num(data[data['spmode']=='998']['BLegendreP2']))
    denom1005 = ak.sum(ak.num(data[data['spmode']=='1005']['BLegendreP2']))
    denom999 = ak.sum(ak.num(data[data['spmode']=='-999']['BLegendreP2']))
    fomL = []
    for i in range(10000):
        if val998[i]+val1005[i]>0:
            fomL += [(val999[i]/denom999)/np.sqrt(((val998[i]/denom998)+(val1005[i]/denom1005))/2)]
        else: fomL += [0]

    print(sum(1 for val in fomL if val == 0))


    maxInd = fomL.index(max(fomL))
    print(max(fomL), maxInd)
    print("BCTS  <",((maxInd%10)/100+0.58))
    maxInd=maxInd//10
    print("BCTT <",((maxInd%10)/100+0.47))
    maxInd=maxInd//10
    print("BP2 <",(maxInd%10)/10+1.2)
    maxInd=maxInd//10
    print("TMA <",((maxInd%10)+79)/100)


    print("998 lambda: ", val998[maxInd])
    print("1005 lambda: ", val1005[maxInd])
    print("-999 lambda: ", val999[maxInd])







for spChoice in ['-999','1005','998']:

    indices_and_maps = get_info_for_PID_masks(data, verbosity=0)

    mask_bool_protonL1, mask_bool_pionL1,mask_bool_protonL2, mask_bool_pionL2 = PID_masks(indices_and_maps, \
                lam1p_selector=myPMask, \
                lam1pi_selector=myPiMask, \
                lam2p_selector=myPMask, \
                lam2pi_selector=myPiMask, \
                verbosity=0)
    
    spmask = (data['spmode']==spChoice)

    mask_pid = mask_bool_pionL1 & mask_bool_protonL1 & mask_bool_pionL2 & mask_bool_protonL2 

    mask_fit = mask_pid[spmask]


    lamconflsig = data[spmask]['Lambda0postFitFlightSignificance'][mask_fit]
    numBCan = data[spmask]['nB']
        

    mask_fl = (lamconflsig>minSigCut)

    
    flightSigMask = (ak.all(mask_fl,axis=1))


    bCanMask = (numBCan==1)

    vetoMask = apply_antibaryon_veto(data, pps, myPMask)
    vetoMask = vetoMask[spmask]

    bcm = data[spmask]['BcosthCM']
    bts = data[spmask]['BCosThetaS']
    btt = data[spmask]['BCosThetaT']
    blp = data[spmask]['BLegendreP2']
    tma = data[spmask]['thrustMagAll']
    r2a = data[spmask]['R2All']

    bcm = bcm[flightSigMask&bCanMask&vetoMask]
    bcm = ak.flatten(bcm, axis=None)
    bcm = ak.to_numpy(bcm)

    if spChoice == '998':
        BM.fill(BcosthCM = bcm, SP1=spChoice,  weight=(424.3/1719))
    elif spChoice == '1005':
        BM.fill(BcosthCM = bcm, SP1=spChoice,  weight=(424.3/868.3))
    elif spChoice == '-999':
        BM.fill(BcosthCM = bcm, SP1=spChoice,  weight=1)

    bts = bts[flightSigMask&bCanMask&vetoMask]
    bts = ak.flatten(bts, axis=None)
    bts = ak.to_numpy(bts)

    if spChoice == '998':
        BS.fill(BCosThetaS = bts, SP2=spChoice,  weight=(424.3/1719))
        x1 = np.append(x1,bts)
    elif spChoice == '1005':
        BS.fill(BCosThetaS = bts, SP2=spChoice,  weight=(424.3/868.3))
        x1 = np.append(x1,bts)
    elif spChoice == '-999':
        BS.fill(BCosThetaS = bts, SP2=spChoice,  weight=1)
        x1S = np.append(x2,bts)

    btt = btt[flightSigMask&bCanMask&vetoMask]

    btt = ak.flatten(btt, axis=None)
    btt = ak.to_numpy(btt)

    if spChoice == '998':
        BT.fill(BCosThetaT = btt, SP3=spChoice,  weight=(424.3/1719))
        x2 = np.append(x2,btt)
    elif spChoice == '1005':
        BT.fill(BCosThetaT = btt, SP3=spChoice,  weight=(424.3/868.3))
        x2 = np.append(x2,btt)
    elif spChoice == '-999':
        BT.fill(BCosThetaT = btt, SP3=spChoice,  weight=1)
        x2S = np.append(x2S,btt)

    blp = blp[flightSigMask&bCanMask&vetoMask]
    blp = ak.flatten(blp, axis=None)
    blp = ak.to_numpy(blp)

    if spChoice == '998':
        B2.fill(BLegendreP2 = blp, SP4=spChoice,  weight=(424.3/1719))
        y1 = np.append(y1,blp)
    elif spChoice == '1005':
        B2.fill(BLegendreP2 = blp, SP4=spChoice,  weight=(424.3/868.3))
        y1 = np.append(y1,blp)
    elif spChoice == '-999':
        B2.fill(BLegendreP2 = blp, SP4=spChoice,  weight=1)
        y1S = np.append(y2,blp)

    tma = tma[flightSigMask&bCanMask&vetoMask]
    tma = ak.flatten(tma, axis=None)
    tma = ak.to_numpy(tma)

    if spChoice == '998':
        TA.fill(thrustMagAll = tma, SP5=spChoice,  weight=(424.3/1719))
        y2 = np.append(y2,tma)
    elif spChoice == '1005':
        TA.fill(thrustMagAll = tma, SP5=spChoice,  weight=(424.3/868.3))
        y2 = np.append(y2,tma)
    elif spChoice == '-999':
        TA.fill(thrustMagAll = tma, SP5=spChoice,  weight=1)
        y2S = np.append(y2S,tma)

    r2a = r2a[flightSigMask&bCanMask&vetoMask]
    r2a = ak.flatten(r2a, axis=None)
    r2a = ak.to_numpy(r2a)

    if spChoice == '998':
        RA.fill(R2All = r2a, SP6=spChoice,  weight=(424.3/1719))
    elif spChoice == '1005':
        RA.fill(R2All = r2a, SP6=spChoice,  weight=(424.3/868.3))
    elif spChoice == '-999':
        RA.fill(R2All = r2a, SP6=spChoice,  weight=1)

isS = True

plt.figure(figsize=(8,6))
BM.stack('SP1')[:].project("BcosthCM").plot(stack=isS,histtype="fill")
plt.legend() 
plt.show()

plt.figure(figsize=(8,6))
BS.stack('SP2')[:].project("BCosThetaS").plot(stack=isS,histtype="fill")
plt.legend() 
plt.show()

plt.figure(figsize=(8,6))
BT.stack('SP3')[:].project("BCosThetaT").plot(stack=isS,histtype="fill")
plt.legend() 
plt.show()

plt.figure(figsize=(8,6))
B2.stack('SP4')[:].project("BLegendreP2").plot(stack=isS,histtype="fill")
plt.legend() 
plt.show()

plt.figure(figsize=(8,6))
TA.stack('SP5')[:].project("thrustMagAll").plot(stack=isS,histtype="fill")
plt.legend() 
plt.show()

plt.figure(figsize=(8,6))
RA.stack('SP6')[:].project("R2All").plot(stack=isS,histtype="fill")
plt.legend() 
plt.show()

import matplotlib.cm as cm

cminVal = 200

fig, ax = plt.subplots(figsize=(8, 6))

bins = (np.linspace(-1.25, 1.25, 51), np.linspace(-1,7.5,51))

counts2, xedges2, yedges2, im2 = ax.hist2d(
    x1S, y1S, 
    bins=bins, 
    cmap="Oranges", 
    cmin=1,
    alpha=1
)

counts1, xedges1, yedges1, im1 = ax.hist2d(
    x1, y1, 
    bins=bins, 
    cmap="Blues", 
    cmin=cminVal, 
    alpha=0.85
)

fig.colorbar(im1, ax=ax, label="998+1005", pad=0.02)
fig.colorbar(im2, ax=ax, label="-999", pad=0.08)

ax.set_xlabel("BCosThetaS")
ax.set_ylabel("BLegendreP2")
ax.set_title("Overlaying 2D Histograms with Distinct Shading")

plt.show()



fig, ax = plt.subplots(figsize=(8, 6))

bins = (np.linspace(-1.25, 1.25, 51), np.linspace(-1,7.5,51))

counts2, xedges2, yedges2, im2 = ax.hist2d(
    x2S, y1S, 
    bins=bins, 
    cmap="Oranges", 
    cmin=1,
    alpha=1
)

counts1, xedges1, yedges1, im1 = ax.hist2d(
    x2, y1, 
    bins=bins, 
    cmap="Blues", 
    cmin=cminVal, 
    alpha=0.85
)

fig.colorbar(im1, ax=ax, label="998+1005", pad=0.02)
fig.colorbar(im2, ax=ax, label="-999", pad=0.08)

ax.set_xlabel("BCosThetaT")
ax.set_ylabel("BLegendreP2")
ax.set_title("Overlaying 2D Histograms with Distinct Shading")

plt.show()



fig, ax = plt.subplots(figsize=(8, 6))

bins = (np.linspace(-1.25, 1.25, 51), np.linspace(0.55,1.05,51))

counts2, xedges2, yedges2, im2 = ax.hist2d(
    x1S, y2S, 
    bins=bins, 
    cmap="Oranges", 
    cmin=1,
    alpha=1
)

counts1, xedges1, yedges1, im1 = ax.hist2d(
    x1, y2, 
    bins=bins, 
    cmap="Blues", 
    cmin=cminVal, 
    alpha=0.85
)

fig.colorbar(im1, ax=ax, label="998+1005", pad=0.02)
fig.colorbar(im2, ax=ax, label="-999", pad=0.08)

ax.set_xlabel("BCosThetaS")
ax.set_ylabel("thrustMagAll")
ax.set_title("Overlaying 2D Histograms with Distinct Shading")

plt.show()



fig, ax = plt.subplots(figsize=(8, 6))

bins = (np.linspace(-1.25, 1.25, 51), np.linspace(0.55,1.05,51))

counts2, xedges2, yedges2, im2 = ax.hist2d(
    x2S, y2S, 
    bins=bins, 
    cmap="Oranges", 
    cmin=1,
    alpha=1
)

counts1, xedges1, yedges1, im1 = ax.hist2d(
    x2, y2, 
    bins=bins, 
    cmap="Blues", 
    cmin=cminVal, 
    alpha=0.85
)

fig.colorbar(im1, ax=ax, label="998+1005", pad=0.02)
fig.colorbar(im2, ax=ax, label="-999", pad=0.08)

ax.set_xlabel("BCosThetaT")
ax.set_ylabel("thrustMagAll")
ax.set_title("Overlaying 2D Histograms with Distinct Shading")

plt.show()


        






"""plt.figure(figsize = (8,6))
plt.hist2d(
        ak.to_numpy(ak.flatten(mesP)), 
        ak.to_numpy(ak.flatten(deP)),
        bins=100, 
        range=[[5.0,5.4], [-0.25,0.25]], 
        cmap='viridis'
    )

plt.colorbar(label='Candidates per bin')
plt.xlabel("m_ES")
plt.ylabel("deltaE")
plt.title('2D m_ES vs DeltaE correlation')"""

"""
# 2. Create a 2D Histogram (highly recommended for large particle physics datasets)
if True:
    plt.figure(figsize=(8, 6))
    plt.hist2d(
        x_combined, 
        y_combined, 
        bins=100, 
        range=[[lo-1.05*width, hi+1.05*width], [lo-1.05*width, hi+1.05*width]], 
        cmap='viridis'
    )

    # 3. Add labels and a colorbar
    plt.colorbar(label='Candidates per bin')
    plt.xlabel(r'$Lambda$ Candidate 1 Mass [GeV/$c^2$]')
    plt.ylabel(r'$Lambda$ Candidate 2 Mass [GeV/$c^2$]')
    plt.title('2D Mass Correlation')

    plt.show()
"""
"""
nsig = len(m[s5Mask]) - b1 - b0

nbkg = b1+b0
npeak = len(m[s5Mask])
nall = len(lamuncmass[lamfl_basic>=0]) #?????? Should this be events with 2 lambda, events with enough fltsig, or all events?

#nall = len(ak.flatten(m[lamfl_basic>=0], axis=None))


#npeak = len(ak.flatten(m[mask], axis=None))
#nbkglo = len(ak.flatten(m[mask_lo_sideband], axis=None))
#nbkghi = len(ak.flatten(m[mask_hi_sideband], axis=None))
#nbkg = (nbkglo + nbkghi)/2.0
#nsig = npeak - nbkg

#print(f"nall: {nall}     npeak: {npeak}   nbkg: {nbkg}     nsig: {nsig}")
peaks.append(npeak)
sigs.append(nsig)
bkgs.append(nbkg)
cuts.append(i)
allentries.append(nall)
tags.append(flight_tag)

row = int(i/ncols)
col = i%ncols

plt.sca(axes1[row][col])
    
plt.hist(ak.flatten(lamuncmass, axis=None),bins=100,range=(1.105, 1.125), label=f'flcut>{i:.1f}')
plt.hist(ak.flatten(m, axis=None),      bins=100,range=(1.105, 1.125))
#plt.hist(ak.flatten(m[mask_lo_sideband], axis=None),bins=100,range=(1.105, 1.125), color='yellow')
#plt.hist(ak.flatten(m[mask_hi_sideband], axis=None),bins=100,range=(1.105, 1.125), color='yellow')
plt.legend()

#plt.sca(axes2[row][col])
if row == nrows-1:
    axes1[row][col].set_xlabel(r'$Lambda^0$ mass [GeV/c$^2$]')
    #axes2[row][col].set_xlabel(r'$Lambda^0$ flight len [cm]')

fllo, flhi = 0,30
if flight_tag=='lampostfitsig':
    fllo, flhi = 0,100

#plt.hist(ak.flatten(lamconflsig, axis=None),         bins=100,range=(fllo, flhi))
#plt.hist(ak.flatten(lamconflsig[mask_fl], axis=None),bins=100,range=(fllo, flhi))

#fig1.subplots_adjust(wspace=0, hspace=0)#left=0.1, right=0.1, bottom=0.1, top=0.1, wspace=0.1, hspace=0.1)#wspace=0, hspace=0)
#fig2.subplots_adjust(wspace=0, hspace=0)#left=0.1, right=0.1, bottom=0.1, top=0.1, wspace=0.1, hspace=0.1)#wspace=0, hspace=0)

#fig1.subplots_adjust(0,0,1,1,0,0)
#fig2.subplots_adjust(0,0,1,1,0,0)

#plt.figure(fig1)
#plt.suptitle(f'{flight_tag}')
#plt.figure(fig2)
#plt.suptitle(f'{flight_tag}')

#fig1.tight_layout()
#fig2.tight_layout()


sigs = np.array(sigs)
bkgs = np.array(bkgs)
peaks = np.array(peaks)
cuts = np.array(cuts)
allentries = np.array(allentries)

pcts = sigs/sigs[0]
bkg_under_peak = peaks - sigs
pct_bkg_under_peak = bkg_under_peak / bkg_under_peak[0]

S = sigs
B = bkg_under_peak
scale = 1
fom = scale*S/np.sqrt(scale*S + scale*B)

max_fom = max(fom)
idx_max = fom.tolist().index(max_fom)
print(valu, "/", myPMask, myPiMask)
print(f"idx_max: {idx_max}  max cuts: {cuts[idx_max]:.2f} max fom: {fom[idx_max]:.2f}   pcts: {pcts[idx_max]:.2f}")


plt.figure(figsize=(12, 8))
plt.subplot(3,2,1)
plt.plot(cuts, peaks,'o')
plt.ylabel('# under peak')
plt.xlabel(f'Cut on flight sig value')

plt.subplot(3,2,2)
plt.plot(cuts, sigs,'o')
#plt.ylabel('# of signal in peak (# in peak - # est background')
plt.ylabel('# of signal in peak')
plt.xlabel(f'Cut on flight sig value')

plt.subplot(3,2,3)
plt.plot(cuts, bkgs,'o')
plt.ylabel('# est background')
plt.xlabel(f'Cut on flight sig value')

plt.subplot(3,2,4)
plt.plot(cuts, pcts,'o')
plt.ylim(0.1,1.1)
plt.ylabel('% sig remaining')
plt.xlabel(f'Cut on flight sig value')

plt.subplot(3,2,5)
plt.plot(cuts, pct_bkg_under_peak,'o')
#plt.ylim(0.7,1.1)
plt.ylabel('% bkg under peak')
plt.xlabel(f'Cut on flight sig value')

# Naive significance
# Multiply by 10 if we are only doing Run 1
plt.subplot(3,2,6)
#plt.plot(cuts, 10*sigs/np.sqrt(10*bkg_under_peak),'o')
plt.plot(cuts, fom,'o')
plt.ylabel(r'$S/sqrt{ S + B }$')
plt.xlabel(f'Cut on flight sig value')

plt.suptitle(f'MYLABEL')

plt.tight_layout()
"""
